# 02. Preprocessing

**Mục tiêu**:
- Xử lý missing values
- Cap outliers
- Parse datetime
- Time-based split (70/15/15)

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.data_loader import load_all_tables
from src.preprocessing import (
    parse_datetime_columns,
    handle_missing_values,
    cap_outliers_iqr,
    time_based_split,
)
from src.utils import set_seed

set_seed(42)
DATA_DIR = Path('../data_raw')
OUT_DIR = Path('../data_processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
tables = load_all_tables(DATA_DIR)
for name, df in tables.items():
    print(name, df.shape)

## 1. Parse datetime columns

In [ ]:
datetime_specs = {
    'trips': ['dispatch_date'],
    'loads': ['booking_date'],
    'delivery_events': ['scheduled_datetime', 'actual_datetime'],
    'fuel_purchases': ['purchase_date'],
    'maintenance_records': ['maintenance_date'],
    'safety_incidents': ['incident_date'],
    'drivers': ['hire_date', 'termination_date'],
    'trucks': ['acquisition_date'],
    'trailers': ['acquisition_date'],
}

for name, cols in datetime_specs.items():
    if name in tables:
        tables[name] = parse_datetime_columns(tables[name], cols)
print('Datetime parsed.')

## 2. Handle missing values

Strategy:
- Numerical → median
- Categorical → 'Unknown'

In [ ]:
for name in tables:
    tables[name] = handle_missing_values(tables[name])
print('Missing values handled.')

## 3. Save processed tables

In [ ]:
for name, df in tables.items():
    df.to_parquet(OUT_DIR / f'{name}.parquet', index=False)
print('All processed tables saved to', OUT_DIR)